In [ ]:
!pip install kaggle pandas numpy scikit-learn joblib matplotlib seaborn
import pandas as pd
import numpy as np
import math
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, precision_recall_curve
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d arashnic/fitbit
!unzip -o fitbit.zip

In [ ]:
# Load data
intensities = pd.read_csv('Fitabase Data 4.12.16-5.12.16/minuteIntensitiesNarrow_merged.csv')
sleep = pd.read_csv('Fitabase Data 4.12.16-5.12.16/minuteSleep_merged.csv')

intensities['ActivityMinute'] = pd.to_datetime(intensities['ActivityMinute'])
sleep['date'] = pd.to_datetime(sleep['date'])

print("Intensities shape:", intensities.shape)
print("Sleep shape:", sleep.shape)
display(intensities.head())
display(sleep.head())

In [ ]:
# Merge and align
df = pd.merge(intensities, sleep, left_on=['Id', 'ActivityMinute'], right_on=['Id', 'date'], how='left')
# Sleep value is 1 (asleep), 2 (restless), 3 (awake). NaN means awake/not tracking.
df['value'] = df['value'].fillna(0)
df['is_asleep'] = (df['value'] == 1).astype(int)
df = df.sort_values(by=['Id', 'ActivityMinute'])

In [ ]:
# Feature engineering: resample to 5-min bins
df.set_index('ActivityMinute', inplace=True)

features_list = []
for pid, user_df in df.groupby('Id'):
    resampled = user_df.resample('5T')
    
    intensity_mean = resampled['Intensity'].mean()
    intensity_std = resampled['Intensity'].std().fillna(0)
    intensity_max = resampled['Intensity'].max()
    label = resampled['is_asleep'].apply(lambda x: 1 if x.median() > 0.5 else 0)
    
    def compute_advanced(series):
        mags = series.values
        if len(mags) <= 1 or np.std(mags) == 0:
            zcr = 0.0
        else:
            mean_centered = mags - np.mean(mags)
            zcr = np.sum(np.diff(np.sign(mean_centered)) != 0) / len(mags)
            
        streak = 0
        for m in reversed(mags):
            if m < 0.05:
                streak += 1
            else:
                break
        return pd.Series({'zcr': zcr, 'streak': streak})
        
    advanced = resampled['Intensity'].apply(compute_advanced)
    
    hour = intensity_mean.index.hour
    minute = intensity_mean.index.minute
    total_minutes = hour * 60 + minute
    hour_sin = np.sin(2 * np.pi * total_minutes / 1440.0)
    hour_cos = np.cos(2 * np.pi * total_minutes / 1440.0)
    
    user_features = pd.DataFrame({
        'Id': pid,
        'accel_magnitude_mean': intensity_mean,
        'accel_magnitude_std': intensity_std,
        'accel_magnitude_max': intensity_max,
        'zero_crossing_rate': advanced['zcr'],
        'notification_delta': 0, # placeholder
        'consecutive_still_count': advanced['streak'],
        'charging': 0, # placeholder
        'hour_sin': hour_sin,
        'hour_cos': hour_cos,
        'label': label
    })
    features_list.append(user_features)

dataset = pd.concat(features_list).dropna()

In [ ]:
# Label distribution
print(dataset['label'].value_counts(normalize=True))

In [ ]:
# Train/test split
X = dataset[['accel_magnitude_mean', 'accel_magnitude_std', 'accel_magnitude_max', 
             'zero_crossing_rate', 'notification_delta', 'consecutive_still_count', 
             'charging', 'hour_sin', 'hour_cos']]
y = dataset['label']
groups = dataset['Id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [ ]:
# Train model
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train, sample_weight=sample_weights)

In [ ]:
# Evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

y_scores = model.predict_proba(X_test)[:, 1]
precision, recall, _ = precision_recall_curve(y_test, y_scores)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.step(recall, precision, where='post')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')

plt.subplot(1, 2, 2)
importances = model.feature_importances_
sns.barplot(x=importances, y=X.columns)
plt.title('Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
# Export
joblib.dump(model, "sleep_model.pkl")
try:
    from google.colab import files
    files.download("sleep_model.pkl")
except ImportError:
    print("Not running in Colab, saved model locally.")